In [1]:
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM,BitsAndBytesConfig


In [2]:
!pip install -U -q "accelerate>=1.1.0"

In [3]:
!pip install -U -q "bitsandbytes>=0.46.1"

In [2]:
# Phải dùng quantization vì model quá lớn k fit vừa 12GB VRAN
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [3]:
model_name = "ehartford/WizardLM-7B-Uncensored"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    dtype=torch.float16
)
model.to("cuda")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id
device = "cuda" if torch.cuda.is_available() else "cpu"

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 86.00 MiB. GPU 0 has a total capacity of 11.59 GiB of which 49.19 MiB is free. Including non-PyTorch memory, this process has 10.98 GiB memory in use. Of the allocated memory 10.80 GiB is allocated by PyTorch, and 12.72 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
def generate_synthetic_data(prompt, max_tokens=256, temperature=0.7):
    # Convert the prompt to tokens and send to GPU/CPU
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # Generate output from model
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=temperature,
        top_p=0.9
    )

    # Decode the generated tokens back to text
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
prompt = """
Generate 5 synthetic job-candidate matching records in JSON format.
Include:
- candidate_id (UUID)
- candidate_name
- current_role
- years_of_experience
- technical_skills (List of strings: languages, frameworks, tools)
- resume_summary (A short paragraph describing professional background)
- applied_job_title
- job_description_requirements (Key requirements from the JD to match against)
- matching_score (A decimal between 0 and 1 representing initial fit)
- missing_skills (List of skills the candidate lacks for this specific JD)
"""

In [ ]:
synthetic_data = generate_synthetic_data(prompt)
print(synthetic_data)

In [ ]:
!pip install gradio -q
import gradio as gr

def generate_ui(prompt):
    return generate_synthetic_data(prompt)

interface = gr.Interface(
    fn=generate_ui,               # function to run
    inputs="text",                # input type is text
    outputs="text",               # output type is text
    title="Synthetic Medical Data Generator",
    description="Enter a prompt to generate synthetic medical dataset in JSON format."
)

# Launch the UI